In [2]:
import cv2
import numpy as np

def load_images(path1, path2, size=(300, 300)):
    img1 = cv2.imread(path1, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(path2, cv2.IMREAD_GRAYSCALE)

    img1 = cv2.resize(img1, size)
    img2 = cv2.resize(img2, size)

    A = img1.astype(np.float64)
    B = img2.astype(np.float64)

    return A, B


def morph_gray_k(A, B, steps=60, k=150):
    frames = []

    UA, SA, VAt = np.linalg.svd(A, full_matrices=False)
    UB, SB, VBt = np.linalg.svd(B, full_matrices=False)

    # sign alignment
    for i in range(UA.shape[1]):
        if np.dot(UA[:, i], UB[:, i]) < 0:
            UB[:, i] = -UB[:, i]
            VBt[i, :] = -VBt[i, :]

    for t in np.linspace(0, 1, steps):
        Ut  = (1-t)*UA[:, :k] + t*UB[:, :k]
        St  = (1-t)*SA[:k]    + t*SB[:k]
        Vtt = (1-t)*VAt[:k,:] + t*VBt[:k,:]

        frame = Ut @ np.diag(St) @ Vtt
        frame = np.clip(frame, 0, 255).astype(np.uint8)

        frames.append(frame)

    return frames


def naive_blend(A, B, steps=60):
    frames = []
    for t in np.linspace(0, 1, steps):
        frame = (1-t)*A + t*B
        frame = np.clip(frame, 0, 255).astype(np.uint8)
        frames.append(frame)
    return frames


A, B = load_images("rigid2.png", "rigid1.png")

k = 150

print("Generating SVD frames...")
svd_forward = morph_gray_k(A, B, steps=60, k=k)
svd_backward = svd_forward[::-1]
svd_frames = svd_forward + svd_backward

print("Generating naive frames...")
naive_forward = naive_blend(A, B, steps=60)
naive_backward = naive_forward[::-1]
naive_frames = naive_forward + naive_backward

print("Done! Press q to quit, +/- to change k")

while True:
    quit_flag = False
    recompute = False

    for svd_frame, naive_frame in zip(svd_frames, naive_frames):

        # convert to 3-channel for display (so we can stack)
        svd_display = cv2.cvtColor(svd_frame, cv2.COLOR_GRAY2BGR)
        naive_display = cv2.cvtColor(naive_frame, cv2.COLOR_GRAY2BGR)

        cv2.putText(svd_display, f'SVD Morph k={k}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.putText(naive_display, 'Naive Blend', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        combined = np.hstack([naive_display, svd_display])
        cv2.imshow("Naive vs SVD Morph", combined)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            quit_flag = True
            break
        elif key == ord('+'):
            k = min(300, k + 50)
            recompute = True
            break
        elif key == ord('-'):
            k = max(1, k - 50)
            recompute = True
            break

    if quit_flag:
        break

    if recompute:
        print(f"Recomputing SVD with k={k}...")
        svd_forward = morph_gray_k(A, B, steps=60, k=k)
        svd_backward = svd_forward[::-1]
        svd_frames = svd_forward + svd_backward

cv2.destroyAllWindows()

Generating SVD frames...
Generating naive frames...
Done! Press q to quit, +/- to change k
